<a href="https://colab.research.google.com/github/arjun-1238/Machine-Learning-Models/blob/main/Logistic_regression_HRdataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [58]:
import pandas as pd

# Load a standard HR attrition dataset directly via URL
url = "https://raw.githubusercontent.com/IBM/employee-attrition-aif360/master/data/emp_attrition.csv"
df = pd.read_csv(url)

In [59]:
print(df.head(4))
display(df.isnull().sum())
print(df["Attrition"].value_counts())

   Age Attrition     BusinessTravel  DailyRate              Department  \
0   41       Yes      Travel_Rarely       1102                   Sales   
1   49        No  Travel_Frequently        279  Research & Development   
2   37       Yes      Travel_Rarely       1373  Research & Development   
3   33        No  Travel_Frequently       1392  Research & Development   

   DistanceFromHome  Education EducationField  EmployeeCount  EmployeeNumber  \
0                 1          2  Life Sciences              1               1   
1                 8          1  Life Sciences              1               2   
2                 2          2          Other              1               4   
3                 3          4  Life Sciences              1               5   

   ...  RelationshipSatisfaction StandardHours  StockOptionLevel  \
0  ...                         1            80                 0   
1  ...                         4            80                 1   
2  ...                  

,0
Age,0
Attrition,0
BusinessTravel,0
DailyRate,0
Department,0
DistanceFromHome,0
Education,0
EducationField,0
EmployeeCount,0
EmployeeNumber,0


Attrition
No     1233
Yes     237
Name: count, dtype: int64


In [60]:
from sklearn.utils import resample

y_for_balancing = df["Attrition"].map({"Yes":1,"No":0})

combined_df = df.copy()
combined_df['Attrition'] = y_for_balancing
df_majority = combined_df[combined_df['Attrition'] == 0]
df_minority = combined_df[combined_df['Attrition'] == 1]


df_majority_downsampled = resample(
    df_majority,
    replace=False,
    n_samples=len(df_minority),
    random_state=42
)

balanced_df = pd.concat([df_majority_downsampled, df_minority])
df_balanced = balanced_df.drop('Attrition', axis=1)
y_balanced = balanced_df['Attrition']

print(y_balanced.value_counts())

Attrition
0    237
1    237
Name: count, dtype: int64


In [61]:
x_category=df_balanced.select_dtypes(include=['object','category'])
x_numerical=df_balanced.select_dtypes(include=['int64','float64'])

df_balanced=pd.get_dummies(df_balanced,columns=x_category.columns,drop_first=True)
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
df_balanced[x_numerical.columns]=scaler.fit_transform(df_balanced[x_numerical.columns])

In [62]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(df_balanced,y_balanced,test_size=0.2,random_state=42,stratify=y_balanced)

In [63]:
from sklearn.linear_model import LogisticRegression
model=LogisticRegression(max_iter=200)
model.fit(x_train,y_train)
y_pred=model.predict(x_test)
from sklearn.metrics import classification_report,accuracy_score,confusion_matrix
print(classification_report(y_test,y_pred))
print(accuracy_score(y_test,y_pred))
print(confusion_matrix(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.69      0.65      0.67        48
           1       0.66      0.70      0.68        47

    accuracy                           0.67        95
   macro avg       0.67      0.67      0.67        95
weighted avg       0.67      0.67      0.67        95

0.6736842105263158
[[31 17]
 [14 33]]
